# **Практика 8. Линейная регрессия.**

[SmartLMS](https://edu.hse.ru/mod/quiz/view.php?id=1918963)

Бар-экзамен — это лицензионный экзамен, который должны сдать выпускники юридических школ в США и некоторых других странах, чтобы получить право заниматься юридической практикой в качестве адвоката. Экзамен проверяет знание законодательства, юридических принципов и этики. Успешная сдача бар-экзамена является обязательным условием для получения лицензии адвоката.

Мы проанализируем данные об абитуриентах и выпускниках юридических школ США, включая их демографические данные, академические показатели (LSAT, GPA), форму обучения и результаты экзамена на адвоката (бар-экзамена).

- `ID`: Уникальный идентификационный номер абитуриента/студента в базе данных.
- `gender`: Пол абитуриента (1- мужчина; 0 - женщина)
- `race`: Раса или этническая принадлежность абитуриента (категориальная переменная, например: White, Black, Asian, Hispanic).
- `DOB_yr`: Год рождения абитуриента (количественная дискретная переменная).
- `lsat`: Результат теста LSAT (Law School Admission Test) — стандартизированный экзамен для поступления в юридическую школу.
- `ugpa`: Средний балл диплома о высшем образовании (Undergraduate GPA) — успеваемость в бакалавриате.
- `fulltime`: Указание на обучение на дневном отделении (фул-тайм).
- `bar_score`: Балл, полученный на экзамене на адвоката (бар-экзамене).
- `index6040`: Составной индекс, рассчитываемый по формуле 0.6LSAT + 0.4UGPA (стандартная формула для оценки абитуриентов в юридических школах).

In [1]:
import warnings
warnings.simplefilter('ignore')

In [2]:
import pandas as pd
df = pd.read_csv('data/bar_pass.csv')
df.head(2)

,ID,gender,race,DOB_yr,lsat,ugpa,fulltime,bar_score,index6040
0,2,1,white,1969,44,3.5,1,323,886.842082
1,3,1,white,1969,29,3.5,1,323,649.999987


## **Задание 1**
Будем считать, что выбросами считаются наблюдения, которые отличаются от среднего арифметического на более чем 2 стандартных отклонения. Определите, сколько наблюдений считаются выбросами по признаку `bar_score`.

>**Таблицу без выбросов сохраните в переменную `df_clean`. Далее во всех заданиях вы работаете с этой таблицей.**

In [3]:
bar_mean = df['bar_score'].mean()
bar_std = df['bar_score'].std()

df_clean = df[(df['bar_score'] > bar_mean - 3 * bar_std) & (df['bar_score'] < bar_mean + 3 * bar_std)]
df.shape[0] - df_clean.shape[0]

326

---

## **Задание 2**

### 2.1
Мы хотим построить модель линейной регрессии, которая будет показывать как **X1** оказывает влияние на результаты экзамена на адвоката (`bar_score`). Известно, что **X1** - это признак, который имеет самую сильную линейную связь с `bar_score`.

Составьте уравнение регрессии:
$$[???] = w_0 + [???] \times w_1$$


In [4]:
df_clean.corr(numeric_only=True)['bar_score'].sort_values(key=abs, ascending=False)

bar_score    1.000000
ugpa         0.630752
index6040    0.360416
lsat         0.146483
DOB_yr       0.103842
gender       0.081525
fulltime     0.075132
ID          -0.002084
Name: bar_score, dtype: float64

**X1** - `ugpa`

**Ответ**:
$$\mathrm{bar\_score} = w_0 + \mathrm{ugpa} \times w_1$$


### 2.2
Соотнесите термины регрессии и составляющие модели:

- Целевая (зависимая) переменная - `bar_score`
- Свободной коэффициент - `w0`
- Коэффициент независимой переменной - `w1`
- Предиктор/независимая переменная - `ugpa`

>*Что является наблюдением в этой модели?*

---

## **Задание 3**
Обучите модель линейной регрессии из прошлого задания и составьте полученное уравнение регрессии. Количественные ответы округлите до двух знаков.

$$[???] = [???] + [???] \times [???]$$

In [5]:
import statsmodels.api as sm

X = sm.add_constant(df_clean['ugpa'])
Y = df_clean['bar_score']

model = sm.OLS(Y, X).fit()
model.params.round(2)

const    296.86
ugpa       8.42
dtype: float64

**Ответ**
$$\mathrm{bar\_score} = 8.42 + \mathrm{ugpa} \times 296.86$$

---

In [6]:
model.params

const    296.859277
ugpa       8.424288
dtype: float64

## **Задание 4**
Проинтерпретируйте полученную модель. Ответ округлите до двух знаков.



In [7]:
lr_func = lambda x: model.params['const'] + x * model.params['ugpa']

print(lr_func(0).round(2))
print(lr_func(1).round(2))
print(lr_func(3).round(2))

296.86
305.28
322.13


- Если предиктор равен 0, то балл за экзамен равен **8.42**
- Если предиктор увеличится на 1 балл, то балл за экзамен увеличится на **296.86**
- Если предиктор равен 1, то балл за экзамен будет равен **305.28**
- Если предиктор равен 3, то балл за экзамен будет равен **322.13**

---

## **Задание 5**
Рассчитайте основные метрики качества модели. Ответы округлите до двух знаков.



In [8]:
Y_pred = model.predict(X)

mae = (Y - Y_pred).abs().mean()
mse = ((Y - Y_pred)**2).mean()
r2 = model.rsquared

print(mae.round(2))
print(mse.round(2))
print((r2 * 100).round(2))

3.66
15.76
39.78


- В среднем модель ошибается в предсказании результатов экзамена на **3.66** баллов
- Уcреденный квадрат ошибки модели, в предсказании результатов экзамена, составляет **15.76** баллов$^2$
- Модель смогла описать **39.78** % вариаций целевой переменной.


---

## **Задание 6**

### 6.1
Постройте сводную таблицу, где по строкам этническая принадлежность (`race`), а в столбце медианный балл экзамена на адвокатскую лицензию (`bar_score`) для каждой этнической принадлежности. Укажите этническую принадлежность (`race`), для которой медианный балл за экзамен наименьший.



In [9]:
df_clean.groupby('race')['bar_score'].median().sort_values()

race
black    321.0
hisp     323.0
other    323.0
asian    324.0
white    324.0
Name: bar_score, dtype: float64

### 6.2
Создайте новый признак `race_recoded` на основе признака `race`. Если этническая принадлежность студента (`race`) совпадает с той, у которой средний балл за экзамен наименьший, вернуть 1, иначе 0. В ответ укажите долю 1 по признаку `race_recoded`. Ответ округлите до двух знаков.



In [10]:
df_clean['race_recoded'] = (df_clean['race'] == 'black').astype(int)
df_clean['race_recoded'].mean().round(2)

0.06

---

## **Задание 7**

### 7.1
Обучите многофакторную линейную регрессию, которая будет предсказывает результаты экзамена (`bar_score`) на основе признака `race_recoded` и `index6040`.
$$\mathrm{bar\_score} = [???] \times \mathrm{race\_recoded} + [???] \times \mathrm{index6040} + [???]$$
Укажите параметры регрессии. Ответы округлите до двух знаков.



In [13]:
X = sm.add_constant(df_clean[['race_recoded', 'index6040']])
Y = df_clean['bar_score']

model_new = sm.OLS(Y, X).fit()
model_new.params.round(2)

const           311.05
race_recoded     -0.74
index6040         0.02
dtype: float64

>*Какое наибольшее количество предикторов, можно включить в нашу модель, если каждый предиктор должен быть обеспечен минимум 100 наблюдениями?*

### 7.2
Проинтерпретируйте полученную модель: 


- Если человек принадлежит к этнической группе с наименьшим медианным баллом бар, то это **снижает** результаты экзамена на **0.74** баллов, при прочих равных.
- Если значение индекса увеличится на 2 пункта, то результаты экзамена увеличатся на **0.04**  балла, при прочих равных.


### 7.3
Рассчитайте метрики качества полученной модели. Ответы округлите до двух знаков.



In [16]:
Y_pred = model_new.predict(X)

mae = (Y - Y_pred).abs().mean()
mse = ((Y - Y_pred)**2).mean()
r2 = model_new.rsquared

print(f'mae: {mae.round(2)}')
print(f'mse: {mse.round(2)}')
print(f'r2: {r2.round(2)}')

mae: 3.96
mse: 22.75
r2: 0.13


>*Проинтерпретируйте полученные метрики качества модели. Сравните модели между собой.*

---